In [1]:
# ==========================================
#   IMPORT LIBRARIES & SETUP
# ==========================================
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'   # implicit recommends this for ALS

%load_ext autoreload
%autoreload 2

from metrics import evaluate
import pandas as pd 
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import CosineRecommender
import itertools
import time

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## DATA LOADING

In [2]:
articles=pd.read_parquet('data/articles_processed.parquet')
customers=pd.read_parquet('data/customers_processed.parquet')
transactions=pd.read_parquet('data/transactions_processed.parquet')


In [3]:
#=================================================================================
#                        MAPPING
#=================================================================================

unique_customers = customers['customer_id'].unique()
unique_articles = articles['article_id'].unique()

# Mappings
customer2idx = {c: i for i, c in enumerate(unique_customers)}
article2idx = {a: i for i, a in enumerate(unique_articles)}

# Reverse Mappings 
idx2customer = {i: c for c, i in customer2idx.items()}
idx2article = {i: a for a, i in article2idx.items()}

# Mapping transactions-ի վրա
transactions['user_idx'] = transactions['customer_id'].map(customer2idx)
transactions['item_idx'] = transactions['article_id'].map(article2idx)

# Check: no unmapped IDs
print(transactions['user_idx'].isna().sum(), transactions['item_idx'].isna().sum())

0 0


In [4]:
#================================================================================
#                  TRAIN-VALIDATION-TEST SPLIT
#================================================================================
last_date = transactions['t_dat'].max()
test_start = last_date - pd.Timedelta(days=6)    # last week
val_start = test_start - pd.Timedelta(days=7)    # the week before last

train = transactions[transactions['t_dat'] < val_start]
val = transactions[(transactions['t_dat'] >= val_start) & (transactions['t_dat'] < test_start)]
test = transactions[transactions['t_dat'] >= test_start]

## Collaborative Filtering

In [5]:
#=================================================================================
#                              CSR Matrix
#=================================================================================

train_user_item = train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')

rows = train_user_item['user_idx'].values
cols = train_user_item['item_idx'].values
weights = train_user_item['weight'].values.astype(np.float32)

n_users = len(customer2idx)
n_items = len(article2idx)


user_item_matrix = csr_matrix((weights, (rows, cols)), shape=(n_users, n_items))


* Այն հաճախորդների համար , ովքեր գնումներ չեն ունեցել կօգտագործենք վերջին շաբաթվա ամենաշատ գնված ապրանքները

In [6]:

recent_popular = train[train['t_dat'] >= train['t_dat'].max() - pd.Timedelta(days=6)]
fallback_top12 = recent_popular.groupby('article_id').size().nlargest(12).index.tolist()

def predict_implicit_model(model, val_df, k=12):
    
    actuals_dict = val_df.groupby('customer_id')['article_id'].unique().apply(set).to_dict()
    val_customers = list(actuals_dict.keys())

    user_idxs = np.array([customer2idx[c_id] for c_id in val_customers])

    has_history = np.asarray(user_item_matrix[user_idxs].getnnz(axis=1)) > 0

    ids, scores = model.recommend(
        user_idxs,
        user_item_matrix[user_idxs],
        N=k,
        filter_already_liked_items=False
    )

    predictions = {}
    for i, c_id in enumerate(val_customers):
        if has_history[i]:
            recs = [idx2article[int(idx)] for idx in ids[i] if idx >= 0]
        else:
            recs = []  # cold customer
        fill = [p for p in fallback_top12 if p not in recs]
        predictions[c_id] = (recs + fill)[:k]

    return predictions, actuals_dict

In [7]:
#=================================================================================
#                   Item-Item Cosine Similarity
#=================================================================================

cosine_model = CosineRecommender(K=20)
cosine_model.fit(user_item_matrix)
print("Cosine Similarity model successfully fitted!")
print(cosine_model.similarity.shape)

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.14708614349365234 seconds
  warnings.warn(
100%|██████████| 105542/105542 [00:05<00:00, 18607.22it/s]


Cosine Similarity model successfully fitted!
(105542, 105542)


In [8]:
# Մոդելի աշխատանքը գնահատելու համար
total_catalog_items = articles['article_id'].nunique()
sales_series = train['article_id'].value_counts()

total_sales = len(train)

novelty_dict = (
    (-np.log2(sales_series / total_sales)).to_dict()
)

In [9]:
cosine_preds, actuals_dict = predict_implicit_model(cosine_model, val, k=12)

cosine_results = evaluate(cosine_preds, actuals_dict, total_catalog_items,novelty_dict, k=12)

print("--- Cosine Similarity Results ---")
for metric_name, value in cosine_results.items():
    print(f"{metric_name}: {value:.5f}")

--- Cosine Similarity Results ---
Precision@12: 0.00497
Recall@12: 0.02654
MAP@12: 0.01075
NDCG@12: 0.01697
Novelty@12: 14.71539
Coverage: 0.69442


In [10]:
val_results = pd.read_csv('results/val_results.csv', index_col=0)

cosine_row = pd.DataFrame(cosine_results, index=['Cosine kNN (K=20)'])
val_results = pd.concat([val_results, cosine_row])

val_results.to_csv('results/val_results.csv')
val_results

,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.002341,0.008971,0.003444,0.006318,10.216682,0.000114
Recent Popularity (7d),0.004933,0.020905,0.006604,0.012605,13.269735,0.000114
Repeat + Popularity Fallback,0.004374,0.023520,0.009352,0.015019,14.318081,0.569129
Top 5 Repeat + Popularity,0.005515,0.026711,0.009696,0.016547,13.521146,0.427062
Cosine kNN (K=20),0.004973,0.026540,0.010754,0.016974,14.715392,0.694415


* Baseline-ը ցույց է տվել , որ հաճախորդի գնումները հաշվի առնելը լավ արդյունք է տալիս։Ստուգենք val set-ում գնված ապրանքները train-ի վերջին 4 շաբաթվա գնումներից են թե ոչ

In [11]:
val_pairs = val[['user_idx', 'item_idx']].drop_duplicates()
all_hist = train[['user_idx', 'item_idx']].drop_duplicates()

cutoff = train['t_dat'].max() - pd.Timedelta(days=27)   # last 4 weeks of train
recent_hist = train.loc[train['t_dat'] >= cutoff, ['user_idx', 'item_idx']].drop_duplicates()

share_all = len(val_pairs.merge(all_hist)) / len(val_pairs)
share_recent = len(val_pairs.merge(recent_hist)) / len(val_pairs)

print(len(val_pairs), round(share_all, 4), round(share_recent, 4))

227910 0.0393 0.0228


### Repeat գնումների վերլուծություն և hybrid մոդել
 Validation շաբաթվա (հաճախորդ, ապրանք) զույգերի միայն 3.93%-ն է հաճախորդը նախկինում գնել train-ի ողջ պատմության մեջ, այսինքն՝ գնումների մոտ 96%-ը նոր ապրանք է։ Եթե հաշվենք միայն train-ի վերջին 4 շաբաթը, կազմում է 2.28% , այսինքն՝ repeat գնումների մոտ 60%-ը գալիս է վերջին 4 շաբաթվա պատմությունից։

Կառուցենք մոդել ,որ հաճախորդի վերջին 4 շաբաթվա ամենից շատ գնած 2 կամ 3 ապրանքը դնում է ցանկի սկզբում, իսկ մնացած տեղերը լրացվում են Cosine-ով


In [12]:
# --- last 4 weeks of train: how many times each customer bought each article ---
cutoff = train['t_dat'].max() - pd.Timedelta(days=27)   # last 4 weeks of train
recent_train = train[train['t_dat'] >= cutoff]

recent_ui = recent_train.groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')
recent_ui = recent_ui.sort_values(['user_idx', 'weight'], ascending=[True, False])

cosine_preds, actuals_dict = predict_implicit_model(cosine_model, val, k=12)

model_results = {}

for n_repeat in [2, 3, 4, 5, 6, 8, 12]:
    top_repeat = recent_ui.groupby('user_idx').head(n_repeat)
    repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

    hybrid_preds = {}
    for c_id, model_recs in cosine_preds.items():
        repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
        fill = [r for r in model_recs if r not in repeat]   # no duplicates
        hybrid_preds[c_id] = (repeat + fill)[:12]

    model_results[f'Hybrid: {n_repeat} recent repeat + Cosine'] = evaluate(
        hybrid_preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
    print(n_repeat, 'done')



val_results = pd.concat([val_results, pd.DataFrame(model_results).T])
val_results.to_csv('results/val_results.csv')
val_results.round(5)

2 done
3 done
4 done
5 done
6 done
8 done
12 done


,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.00234,0.00897,0.00344,0.00632,10.21668,0.00011
Recent Popularity (7d),0.00493,0.02091,0.00660,0.01260,13.26973,0.00011
Repeat + Popularity Fallback,0.00437,0.02352,0.00935,0.01502,14.31808,0.56913
Top 5 Repeat + Popularity,0.00551,0.02671,0.00970,0.01655,13.52115,0.42706
Cosine kNN (K=20),0.00497,0.02654,0.01075,0.01697,14.71539,0.69442
Hybrid: 2 recent repeat + Cosine,0.00603,0.03300,0.01887,0.02583,14.74163,0.69046
Hybrid: 3 recent repeat + Cosine,0.00643,0.03523,0.01986,0.02733,14.75861,0.68822
Hybrid: 4 recent repeat + Cosine,0.00679,0.03725,0.02049,0.02841,14.77520,0.68618
Hybrid: 5 recent repeat + Cosine,0.00707,0.03855,0.02083,0.02908,14.78923,0.68380
Hybrid: 6 recent repeat + Cosine,0.00732,0.03969,0.02109,0.02960,14.80158,0.68175


In [13]:
#================================================================================
#                              ALS
#================================================================================


als_matrix = user_item_matrix.copy()
als_matrix.data = 1 + np.log(als_matrix.data)

grid = {
    'factors': [64],                      
    'regularization': [0.1, 0.01, 0.3],
    'alpha': [5, 10, 15],
}

path = 'results/als_grid_val.csv'
os.makedirs('results', exist_ok=True)

if os.path.exists(path):
    grid_results = pd.read_csv(path, index_col=0).to_dict('index')
else:
    grid_results = {}

for f, r, a in itertools.product(grid['factors'], grid['regularization'], grid['alpha']):
    name = f'ALS f={f} reg={r} alpha={a}'
    if name in grid_results:
        continue

    t0 = time.time()
    model = AlternatingLeastSquares(factors=f, regularization=r, alpha=a,
                                    iterations=15, random_state=42)
    model.fit(als_matrix, show_progress=False)

    preds, actuals_dict = predict_implicit_model(model, val, k=12)
    metrics = evaluate(preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
    metrics['minutes'] = round((time.time() - t0) / 60, 1)

    grid_results[name] = metrics
    pd.DataFrame.from_dict(grid_results, orient='index').to_csv(path)   # save after every experiment
    print(name, '| MAP@12:', round(metrics['MAP@12'], 5), '| Coverage:', round(metrics['Coverage'], 4))

pd.DataFrame.from_dict(grid_results, orient='index').sort_values('MAP@12', ascending=False).round(5)

ALS f=64 reg=0.1 alpha=5 | MAP@12: 0.00894 | Coverage: 0.018
ALS f=64 reg=0.1 alpha=10 | MAP@12: 0.00865 | Coverage: 0.0256
ALS f=64 reg=0.1 alpha=15 | MAP@12: 0.00846 | Coverage: 0.0315
ALS f=64 reg=0.01 alpha=5 | MAP@12: 0.00894 | Coverage: 0.018
ALS f=64 reg=0.01 alpha=10 | MAP@12: 0.00864 | Coverage: 0.0256
ALS f=64 reg=0.01 alpha=15 | MAP@12: 0.00845 | Coverage: 0.0315
ALS f=64 reg=0.3 alpha=5 | MAP@12: 0.00893 | Coverage: 0.018
ALS f=64 reg=0.3 alpha=10 | MAP@12: 0.00864 | Coverage: 0.0257
ALS f=64 reg=0.3 alpha=15 | MAP@12: 0.00849 | Coverage: 0.0316


,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage,minutes
ALS f=64 reg=0.1 alpha=5,0.00476,0.02135,0.00894,0.01482,11.97630,0.01801,1.9
ALS f=64 reg=0.01 alpha=5,0.00476,0.02135,0.00894,0.01482,11.97562,0.01801,1.8
ALS f=64 reg=0.3 alpha=5,0.00476,0.02134,0.00893,0.01481,11.97697,0.01802,1.9
ALS f=64 reg=0.1 alpha=10,0.00470,0.02116,0.00865,0.01446,12.05772,0.02563,1.8
ALS f=64 reg=0.01 alpha=10,0.00470,0.02113,0.00864,0.01445,12.05620,0.02562,1.8
ALS f=64 reg=0.3 alpha=10,0.00469,0.02112,0.00864,0.01444,12.05919,0.02567,1.8
ALS f=64 reg=0.3 alpha=15,0.00467,0.02116,0.00849,0.01428,12.11911,0.03155,1.8
ALS f=64 reg=0.1 alpha=15,0.00467,0.02114,0.00846,0.01425,12.11730,0.03149,1.8
ALS f=64 reg=0.01 alpha=15,0.00468,0.02115,0.00845,0.01424,12.11548,0.03149,2.1


In [14]:

als_grid = pd.read_csv('results/als_grid_val.csv', index_col=0)

best_name = als_grid['MAP@12'].idxmax()                 # experiment with the highest MAP@12
als_row = als_grid.loc[[best_name]].drop(columns='minutes')
print('Best ALS:', best_name)

val_results = pd.read_csv('results/val_results.csv', index_col=0)
val_results = pd.concat([val_results, als_row])

val_results.to_csv('results/val_results.csv')
val_results.round(5)

Best ALS: ALS f=64 reg=0.1 alpha=5


,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.00234,0.00897,0.00344,0.00632,10.21668,0.00011
Recent Popularity (7d),0.00493,0.02091,0.00660,0.01260,13.26973,0.00011
Repeat + Popularity Fallback,0.00437,0.02352,0.00935,0.01502,14.31808,0.56913
Top 5 Repeat + Popularity,0.00551,0.02671,0.00970,0.01655,13.52115,0.42706
Cosine kNN (K=20),0.00497,0.02654,0.01075,0.01697,14.71539,0.69442
Hybrid: 2 recent repeat + Cosine,0.00603,0.03300,0.01887,0.02583,14.74163,0.69046
Hybrid: 3 recent repeat + Cosine,0.00643,0.03523,0.01986,0.02733,14.75861,0.68822
Hybrid: 4 recent repeat + Cosine,0.00679,0.03725,0.02049,0.02841,14.77520,0.68618
Hybrid: 5 recent repeat + Cosine,0.00707,0.03855,0.02083,0.02908,14.78923,0.68380
Hybrid: 6 recent repeat + Cosine,0.00732,0.03969,0.02109,0.02960,14.80158,0.68175


In [15]:
#==================================================================================
#                 ALS with time-decay weights
#==================================================================================


age_days = (train['t_dat'].max() - train['t_dat']).dt.days   

decay_results = {}
for half_life in [7, 14, 30]:
    w = 0.5 ** (age_days / half_life)
    ui = (train.assign(w=w)
            .groupby(['user_idx', 'item_idx'])['w'].sum()
            .reset_index())

    decay_matrix = csr_matrix(
        (ui['w'].values.astype(np.float32), (ui['user_idx'].values, ui['item_idx'].values)),
        shape=(n_users, n_items))

    model = AlternatingLeastSquares(factors=64, regularization=0.01, alpha=5,
                                    iterations=15, random_state=42)
    model.fit(decay_matrix, show_progress=False)

    preds, actuals_dict = predict_implicit_model(model, val, k=12)
    decay_results[f'ALS time-decay half_life={half_life}'] = evaluate(
        preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
    print(half_life, 'done')

pd.DataFrame(decay_results).T.round(5)

7 done
14 done
30 done


,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
ALS time-decay half_life=7,0.00863,0.04122,0.01820,0.02842,13.89479,0.00840
ALS time-decay half_life=14,0.00877,0.04203,0.01833,0.02876,13.73522,0.00908
ALS time-decay half_life=30,0.00805,0.03850,0.01676,0.02637,13.35762,0.01079


* ALS-ը ժամանակային կշիռներով MAP-ը կրկնապատկեց , բայց այն կրկին անհատական չէ, քանի որ Coverage-ը շատ ցածր է 

In [ ]:
#===================================================================================
# Hybrid: N recent repeat + time-decay ALS (half_life=14)
#===================================================================================

# --- time-decay ALS, half_life=14 ---
age_days = (train['t_dat'].max() - train['t_dat']).dt.days
w = 0.5 ** (age_days / 14)
ui = train.assign(w=w).groupby(['user_idx', 'item_idx'])['w'].sum().reset_index()

decay_matrix = csr_matrix(
    (ui['w'].values.astype(np.float32), (ui['user_idx'].values, ui['item_idx'].values)),
    shape=(n_users, n_items))

als_decay = AlternatingLeastSquares(factors=64, regularization=0.01, alpha=5,
                                    iterations=15, random_state=42)
als_decay.fit(decay_matrix, show_progress=False)

als_preds, actuals_dict = predict_implicit_model(als_decay, val, k=12)

model_results = {
    'Time-decay ALS (half_life=14)': evaluate(als_preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
}

# --- hybrid: recent repeat first, ALS fills the rest ---
for n_repeat in [2, 3, 4, 5, 6, 8, 12]:
    top_repeat = recent_ui.groupby('user_idx').head(n_repeat)
    repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

    hybrid_preds = {}
    for c_id, model_recs in als_preds.items():
        repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
        fill = [r for r in model_recs if r not in repeat]
        hybrid_preds[c_id] = (repeat + fill)[:12]

    model_results[f'Repeat+ALS: {n_repeat} recent repeat + ALS decay'] = evaluate(
        hybrid_preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
    print(n_repeat, 'done')


val_results = pd.read_csv('results/val_results.csv', index_col=0)
val_results = val_results[~(val_results.index.str.startswith('Repeat+ALS') |
                            val_results.index.str.startswith('Time-decay'))]
val_results = pd.concat([val_results, pd.DataFrame(model_results).T])
val_results.to_csv('results/val_results.csv')
val_results.round(5)

2 done
3 done
4 done
5 done
6 done
8 done
12 done


,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.00234,0.00897,0.00344,0.00632,10.21668,0.00011
Recent Popularity (7d),0.00493,0.02091,0.00660,0.01260,13.26973,0.00011
Repeat + Popularity Fallback,0.00437,0.02352,0.00935,0.01502,14.31808,0.56913
Top 5 Repeat + Popularity,0.00551,0.02671,0.00970,0.01655,13.52115,0.42706
Cosine kNN (K=20),0.00497,0.02654,0.01075,0.01697,14.71539,0.69442
Hybrid: 2 recent repeat + Cosine,0.00603,0.03300,0.01887,0.02583,14.74163,0.69046
Hybrid: 3 recent repeat + Cosine,0.00643,0.03523,0.01986,0.02733,14.75861,0.68822
Hybrid: 4 recent repeat + Cosine,0.00679,0.03725,0.02049,0.02841,14.77520,0.68618
Hybrid: 5 recent repeat + Cosine,0.00707,0.03855,0.02083,0.02908,14.78923,0.68380
Hybrid: 6 recent repeat + Cosine,0.00732,0.03969,0.02109,0.02960,14.80158,0.68175


In [17]:
#==================================================================================
# Repeat + ALS: which history window works best for recent repeat?
#==================================================================================


window_results = {}
for weeks in [1, 2, 3, 4, 6, 8]:
    cutoff = train['t_dat'].max() - pd.Timedelta(days=7 * weeks - 1)   
    ui = (train[train['t_dat'] >= cutoff]
        .groupby(['user_idx', 'item_idx']).size().reset_index(name='weight')
        .sort_values(['user_idx', 'weight'], ascending=[True, False]))

    for n_repeat in [3, 5]:
        top_repeat = ui.groupby('user_idx').head(n_repeat)
        repeat_by_user = top_repeat.groupby('user_idx')['item_idx'].apply(list).to_dict()

        hybrid_preds = {}
        for c_id, model_recs in als_preds.items():
            repeat = [idx2article[int(i)] for i in repeat_by_user.get(customer2idx[c_id], [])]
            fill = [r for r in model_recs if r not in repeat]  
            hybrid_preds[c_id] = (repeat + fill)[:12]

        window_results[f'window={weeks}w N={n_repeat}'] = evaluate(
            hybrid_preds, actuals_dict, total_catalog_items, novelty_dict, k=12)
    print(weeks, 'weeks done')

pd.DataFrame(window_results).T[['MAP@12', 'Recall@12', 'Coverage']].round(5)

1 weeks done
2 weeks done
3 weeks done
4 weeks done
6 weeks done
8 weeks done


,MAP@12,Recall@12,Coverage
window=1w N=3,0.02500,0.05031,0.08274
window=1w N=5,0.02547,0.05174,0.09362
window=2w N=3,0.02507,0.05119,0.10652
window=2w N=5,0.02564,0.05316,0.12130
window=3w N=3,0.02468,0.05135,0.12377
window=3w N=5,0.02523,0.05336,0.14132
window=4w N=3,0.02410,0.05121,0.13686
window=4w N=5,0.02462,0.05332,0.15628
window=6w N=3,0.02260,0.05012,0.15525
window=6w N=5,0.02305,0.05230,0.17820


In [18]:
print(pd.DataFrame(window_results).T[['MAP@12', 'Recall@12', 'Coverage']].round(5))

                MAP@12  Recall@12  Coverage
window=1w N=3  0.02500    0.05031   0.08274
window=1w N=5  0.02547    0.05174   0.09362
window=2w N=3  0.02507    0.05119   0.10652
window=2w N=5  0.02564    0.05316   0.12130
window=3w N=3  0.02468    0.05135   0.12377
window=3w N=5  0.02523    0.05336   0.14132
window=4w N=3  0.02410    0.05121   0.13686
window=4w N=5  0.02462    0.05332   0.15628
window=6w N=3  0.02260    0.05012   0.15525
window=6w N=5  0.02305    0.05230   0.17820
window=8w N=3  0.02138    0.04921   0.16719
window=8w N=5  0.02169    0.05114   0.19336


In [19]:
# Add the chosen final model to the validation table

final_row = pd.DataFrame(
    {'Final: 2w N=5 + ALS decay (hl=14)': window_results['window=2w N=5']}
).T

val_results = pd.read_csv('results/val_results.csv', index_col=0)
val_results = val_results[~val_results.index.str.startswith('Final: 2w N=5 + ALS')]   # safe to re-run
val_results = pd.concat([val_results, final_row])

val_results.to_csv('results/val_results.csv')
val_results.round(5)

,Precision@12,Recall@12,MAP@12,NDCG@12,Novelty@12,Coverage
Global Popularity,0.00234,0.00897,0.00344,0.00632,10.21668,0.00011
Recent Popularity (7d),0.00493,0.02091,0.00660,0.01260,13.26973,0.00011
Repeat + Popularity Fallback,0.00437,0.02352,0.00935,0.01502,14.31808,0.56913
Top 5 Repeat + Popularity,0.00551,0.02671,0.00970,0.01655,13.52115,0.42706
Cosine kNN (K=20),0.00497,0.02654,0.01075,0.01697,14.71539,0.69442
Hybrid: 2 recent repeat + Cosine,0.00603,0.03300,0.01887,0.02583,14.74163,0.69046
Hybrid: 3 recent repeat + Cosine,0.00643,0.03523,0.01986,0.02733,14.75861,0.68822
Hybrid: 4 recent repeat + Cosine,0.00679,0.03725,0.02049,0.02841,14.77520,0.68618
Hybrid: 5 recent repeat + Cosine,0.00707,0.03855,0.02083,0.02908,14.78923,0.68380
Hybrid: 6 recent repeat + Cosine,0.00732,0.03969,0.02109,0.02960,14.80158,0.68175
